# BDL Capacity Planning - SHAP Explainability Analysis
**Bharat Dynamics Limited (BDL) - Ministry of Defence, Government of India**

This notebook generates the SHAP (SHapley Additive exPlanations) explainability plots for BDL's capacity planning models. SHAP values are critical for defence-sensitive on-premise systems as they provide full traceability and transparency into model predictions, answering why specific work centers are flagged as bottlenecks.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import os
from train_pipeline import preprocess_and_feature_engineer

# Set seed
np.random.seed(42)

# Load model and preprocessors
with open('models/model_C.pkl', 'rb') as f:
    model_C = pickle.load(f)
with open('models/model_B.pkl', 'rb') as f:
    model_B = pickle.load(f)
with open('models/preprocessors.pkl', 'rb') as f:
    preprocessors = pickle.load(f)

# Load dataset
df = pd.read_csv('data/bdl_production_planning_data.csv')
df_processed, _ = preprocess_and_feature_engineer(df, is_train=False, preprocessors=preprocessors)

# Features list
features_A = preprocessors['categorical_cols'] + [
    c for c in preprocessors['numerical_cols'] if c not in ['load_per_line']
]
features_other = features_A + ['required_capacity_hrs', 'load_per_line']
X = df_processed[features_other]

print("Loaded model and preprocessed data.")

## 1. SHAP Beeswarm Plot (Overall Feature Importance)
This plot shows the top features driving the bottleneck flag prediction across the BDL shop floors.

In [ ]:
explainer_C = shap.TreeExplainer(model_C)
shap_sample = X.sample(500, random_state=42)
shap_values_C = explainer_C(shap_sample)

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values_C, max_display=15)
plt.title('SHAP Feature Importance (Beeswarm) - Bottleneck Flag Classifier', fontsize=14, fontweight='bold')
plt.show()

## 2. SHAP Force Plot: Explaining a Specific Nag ATGM Seeker Assembly Bottleneck
We explain why a specific record representing `Nag` ATGM planning at `WC_SEEKER` with `planning_period_qty=18`, `fai_required=True`, `machine_oee=0.61`, and `num_parallel_lines=1` is flagged as a bottleneck.

In [ ]:
nag_record = {
    'work_center_code': 'WC_SEEKER',
    'machine_code': 'SEEKER_TEST_01',
    'operation_time_min': 300.0,
    'setup_time_min': 60.0,
    'process_sheet_type': 'FAI',
    'pgl_no': 'PGL_NAG_ATGM',
    'operation_sequence': 5,
    'total_smh': 90.0,
    'cost_center': 'CC_HYD',
    'weapon_system': 'Nag',
    'sub_assembly_stage': 'Seeker_Assy',
    'manufacturing_unit': 'Hyderabad',
    'contract_order_qty': 200,
    'planning_period_qty': 18,
    'contracted_delivery_days': 60,
    'delivery_urgency_score': 1.0 / 61.0,
    'available_machine_hrs_day': 8.0,
    'shift_pattern': 'Single',
    'num_parallel_lines': 1,
    'machine_oee_pct': 0.61,
    'machine_age_years': 22,
    'planned_downtime_hrs_month': 15.0,
    'skilled_tech_available': 2,
    'fai_required_flag': 1,
    'qa_gate_clearance_hrs': 4.0,
    'rework_rate_pct': 0.05,
    'drdo_signoff_required': 1,
    'vendor_lead_time_days': 120,
    'indigenisation_pct': 85.0,
    'export_order_flag': 0,
    'required_capacity_hrs': 207.6,
    'utilization_rate': 1.038,
    'bottleneck_flag': 1,
    'delivery_risk_flag': 0,
    'overload_severity': 'Critical'
}

nag_df = pd.DataFrame([nag_record])
nag_processed, _ = preprocess_and_feature_engineer(nag_df, is_train=False, preprocessors=preprocessors)
nag_processed['required_capacity_hrs'] = 207.6
nag_processed['load_per_line'] = 207.6 / 1.0

X_nag = nag_processed[features_other]
shap_nag = explainer_C(X_nag)

# Display Force Plot
shap.initjs()
shap.plots.force(shap_nag[0])

## 3. SHAP Dependence Plot: Capacity Utilization vs Machine OEE
We evaluate how the model utilizes the `machine_oee_pct` feature and see if there are visual differences colored by the specific weapon system's routing characteristics.

In [ ]:
explainer_B = shap.TreeExplainer(model_B)
shap_values_B = explainer_B(shap_sample)

oee_idx = features_other.index('machine_oee_pct')
oee_vals = shap_sample['machine_oee_pct']
shap_oee = shap_values_B.values[:, oee_idx]
weapon_vals = shap_sample['weapon_system']
weapon_names = preprocessors['encoders']['weapon_system'].inverse_transform(weapon_vals)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=oee_vals, y=shap_oee, hue=weapon_names, palette='tab10', alpha=0.8)
plt.title('SHAP Dependence: Utilization vs Machine OEE (colored by Weapon System)', fontsize=14, fontweight='bold')
plt.xlabel('Machine OEE %')
plt.ylabel('SHAP Value (Contribution to Utilization)')
plt.legend(title='Weapon System', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()